# 03 — Apriori Baseline
Association rule mining on Instacart order baskets, with leave-one-out evaluation against 1,000 held-out test users.

In [ ]:
# Install mlxtend if not already installed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "mlxtend", "-q"], check=True)

In [ ]:
import numpy as np
import pandas as pd
import json
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [ ]:
# Hyperparameters — adjust here only
MIN_SUPPORT    = 0.01
MIN_CONFIDENCE = 0.2

# Paths
DATA_DIR = "../data/processed"

In [ ]:
# Load order data
orders = pd.read_csv(f"{DATA_DIR}/orders_subset.csv")
order_products = pd.read_csv(f"{DATA_DIR}/order_products_subset.csv")

# Filter to 'prior' orders only (training baskets — not the test/eval split)
prior_orders = orders[orders["eval_set"] == "prior"][["order_id"]]

# Join to get product lists for each prior order
prior_order_products = prior_orders.merge(order_products, on="order_id")

print(f"Prior orders:    {prior_orders.shape[0]:,}")
print(f"Prior products:  {prior_order_products.shape[0]:,}")
print(prior_order_products.head())

In [ ]:
# Group by order_id → list of product_ids (as strings for mlxtend)
baskets = (
    prior_order_products
    .groupby("order_id")["product_id"]
    .apply(lambda x: [str(p) for p in x.tolist()])
    .tolist()
)

print(f"Total baskets: {len(baskets):,}")
print(f"Sample basket (first 5 items): {baskets[0][:5]}")

# One-hot encode using TransactionEncoder
te = TransactionEncoder()
te_array = te.fit_transform(baskets)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

print(f"Transaction matrix shape: {basket_df.shape}")

In [ ]:
# Mine frequent itemsets
print(f"Mining with min_support={MIN_SUPPORT}, min_confidence={MIN_CONFIDENCE} ...")
frequent_itemsets = apriori(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print(f"Frequent itemsets found: {len(frequent_itemsets):,}")
print(frequent_itemsets.sort_values("support", ascending=False).head(10))

# Generate association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules.sort_values("confidence", ascending=False).reset_index(drop=True)

print(f"\nAssociation rules mined: {len(rules):,}")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

## Evaluation: Leave-One-Out Protocol
Using the 1,000 held-out test users from X_test.npz.
Protocol: query = all prior items in user's sequence, ground truth = last item in the sequence.

In [ ]:
# Load test sequences
test_data = np.load(f"{DATA_DIR}/X_test.npz")
X_test = test_data["sequences"]   # shape: (1000, 100), int32, PAD=0

print(f"X_test shape: {X_test.shape}, dtype: {X_test.dtype}")

# Load vocabulary: product_id (str) → index (int)
with open(f"{DATA_DIR}/vocab.json", "r") as f:
    vocab = json.load(f)

# Build reverse vocab: index (int) → product_id (str)
idx_to_product = {v: k for k, v in vocab.items()}

print(f"Vocab size: {len(vocab):,} entries (including PAD at index 0)")
print(f"Sample reverse mapping: idx 1 → product {idx_to_product.get(1, 'N/A')}")

In [ ]:
def apriori_recommend(query_products, rules_df, top_k=10):
    """
    Given a set of product_id strings (query_products),
    find all rules whose antecedents are a subset of the query,
    collect consequent product IDs ranked by confidence desc,
    return top_k unique consequent IDs not already in query.

    Args:
        query_products: set of product_id strings (str)
        rules_df: DataFrame with 'antecedents', 'consequents', 'confidence' columns
        top_k: int, number of recommendations to return

    Returns:
        list of product_id strings (str), length <= top_k
    """
    query_set = set(str(p) for p in query_products)
    
    # Filter rules where antecedents ⊆ query
    firing_rules = rules_df[
        rules_df["antecedents"].apply(lambda ant: ant.issubset(query_set))
    ].sort_values("confidence", ascending=False)
    
    # Collect consequents in confidence order, exclude items already in query
    recommendations = []
    seen = set(query_set)
    for _, row in firing_rules.iterrows():
        for item in row["consequents"]:
            if item not in seen:
                recommendations.append(item)
                seen.add(item)
        if len(recommendations) >= top_k:
            break
    
    return recommendations[:top_k]

In [ ]:
# Sanity check: use arbitrary top products from frequent_itemsets
top_product = list(frequent_itemsets.sort_values("support", ascending=False).iloc[0]["itemsets"])[0]
test_recs = apriori_recommend({top_product}, rules, top_k=5)
print(f"Sample recs for product {top_product}: {test_recs}")

In [ ]:
def decode_sequence(seq, idx_to_product):
    """Convert int sequence to list of product_id strings, dropping PAD (index 0)."""
    return [idx_to_product[idx] for idx in seq if idx != 0]

# Evaluation
hits_at_5  = 0
hits_at_10 = 0
precision_at_5_total = 0.0
mrr_total = 0.0
N_USERS = X_test.shape[0]  # 1000
skipped = 0

for i in range(X_test.shape[0]):
    seq = X_test[i]                              # shape (100,), int32
    products = decode_sequence(seq, idx_to_product)  # non-PAD items in order

    if len(products) < 2:
        # Not enough items for leave-one-out; skip this user
        skipped += 1
        continue

    # Leave-one-out: last item = ground truth, rest = query
    ground_truth = set([products[-1]])           # last product in sequence
    query = set(products[:-1])                   # all prior products

    # Get recommendations
    recs_10 = apriori_recommend(query, rules, top_k=10)
    recs_5  = recs_10[:5]

    # HR@5
    if any(r in ground_truth for r in recs_5):
        hits_at_5 += 1

    # HR@10
    if any(r in ground_truth for r in recs_10):
        hits_at_10 += 1

    # Precision@5 = hits in top-5 / 5
    hits_in_5 = sum(1 for r in recs_5 if r in ground_truth)
    precision_at_5_total += hits_in_5 / 5

    # MRR = 1 / rank of first hit (0 if no hit)
    for rank, rec in enumerate(recs_10, start=1):
        if rec in ground_truth:
            mrr_total += 1.0 / rank
            break

evaluated_users = X_test.shape[0] - skipped
HR5         = hits_at_5 / evaluated_users
HR10        = hits_at_10 / evaluated_users
PRECISION5  = precision_at_5_total / evaluated_users
MRR         = mrr_total / evaluated_users

print("=" * 45)
print(f"{'Metric':<20} {'Value':>10}")
print("=" * 45)
print(f"{'HR@5':<20} {HR5:>10.4f}")
print(f"{'HR@10':<20} {HR10:>10.4f}")
print(f"{'Precision@5':<20} {PRECISION5:>10.4f}")
print(f"{'MRR':<20} {MRR:>10.4f}")
print("=" * 45)
print(f"\nEvaluated on {evaluated_users} test users (skipped {skipped} with <2 items)")
print(f"Expected HR@5 ≈ 0.33 per PRD")

In [ ]:
# Save results to CSV — one row per metric, columns: metric, value
baseline_results = pd.DataFrame([
    {"metric": "HR@5",        "value": HR5},
    {"metric": "HR@10",       "value": HR10},
    {"metric": "Precision@5", "value": PRECISION5},
    {"metric": "MRR",         "value": MRR},
])

save_path = f"{DATA_DIR}/baseline_results.csv"
baseline_results.to_csv(save_path, index=False)

print(f"Saved: {save_path}")
print(baseline_results.to_string(index=False))

## Apriori Baseline: Limitations and LSTM Improvements

While Apriori association rule mining provides a useful and interpretable baseline for product recommendation, it carries several structural limitations that make it a poor choice for personalized, real-time grocery recommendations. Each limitation is discussed below, along with how the LSTM-based model in this project addresses it.

### 1. Ignores Item Order

Apriori treats each transaction as an unordered set of items, completely discarding the sequence in which a user added products to their basket or across multiple shopping sessions. This means it cannot learn that someone who buys coffee and a coffee maker is likely next to buy filters — it only sees co-occurrence, not causality or progression. The resulting rules reflect population-level buying patterns rather than any individual's purchasing journey. The LSTM model, by contrast, processes item sequences in order, allowing it to capture directional purchasing patterns and predict what a user is likely to buy *next* given their recent history.

### 2. Ignores Temporal Context

Association rules mined from historical baskets carry no information about *when* a purchase is likely to happen. Shoppers behave differently on a Monday morning versus a Sunday evening — cereal and milk spike in morning hours, while snacks and beverages peak on weekends — but Apriori cannot incorporate this signal. Rules that fire in all contexts necessarily sacrifice precision for generality. The Time-Aware LSTM in Phase 5 addresses this by embedding hour-of-day, day-of-week, and days-since-last-order directly into the model, enabling recommendations that shift naturally with the user's current shopping context.

### 3. No Personalization

Apriori produces a single global ruleset applied identically to every user. A seasoned home cook and a college student buying instant noodles receive the same recommendations as long as their current basket looks similar. This means the model can never surface the genuinely personal preferences that differentiate individual shoppers. The LSTM model is trained on each user's own order history and encodes user-specific sequential patterns into its hidden state, producing recommendations that are meaningfully personalized to individuals rather than averaged across the population.

### 4. Cold Start Failure

When a user has no prior order history, Apriori has nothing to query against — its rules require at least some items in the current basket to fire, and even then they reflect population averages rather than the new user's preferences. This is a critical failure mode for any production-grade recommendation system. FreshCart's three-tier cold start system (Phase 6) directly solves this: Tier 1 users (zero orders) receive globally popular items, Tier 2 users (1–2 orders) receive aisle-affinity recommendations, and only Tier 3 users (3+ orders) get personalized LSTM predictions — ensuring no user ever sees an empty recommendation panel.

### 5. No Session Modeling

Apriori operates on completed, static order baskets — it cannot model the dynamics of an *ongoing* shopping session where items are added incrementally and recommendations should update in real time. A rule mined from completed historical checkouts cannot reason about an in-progress cart where only 3 items have been added so far. The LSTM, combined with the live `/api/recommend` endpoint in Phase 8, re-scores recommendations after every cart change, treating the current session as a live sequence and surfacing contextually relevant suggestions as the user shops.

---

*These limitations collectively motivate the project's three-pronged innovation: sequential LSTM modeling (addresses 1, 5), time-aware embeddings (addresses 2), and the three-tier cold start system (addresses 4). Personalization (3) is an emergent property of the LSTM's per-user sequence encoding.*